In [1]:
import numpy as np
print(np.__version__)

1.26.4


In [4]:
!pip install numpy

Defaulting to user installation because normal site-packages is not writeable
    sys-platform (=="darwin") ; extra == 'objc'
                 ~^

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import torch
import numpy as np

print("Torch:", torch.__version__)
print("NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.1.0+cu121
NumPy: 1.26.4
CUDA available: True


In [3]:
torch.cuda.get_device_name(0)
!pip install numpy==1.26.4

Defaulting to user installation because normal site-packages is not writeable
    sys-platform (=="darwin") ; extra == 'objc'
                 ~^

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
# ==============================
# YOLO11n SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLO11n SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11n-seg model...")
model = YOLO("yolo11n-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolo11n_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision (good for RTX 5000 Ada)
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolo11n_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolo11n_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLO11n SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11n-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynam

/home/ranjan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11/yolo11n_seg_training
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/200      2.18G      2.438      3.613      3.304      1.449          0        221        640: 100% ━━━━━━━━━━━━ 63/63 6.5it/s 9.7s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 6.9it/s 0.6s0.2s
                   all         49        694     0.0111      0.264     0.0189    0.00511     0.0111      0.262     0.0287    0.00673

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      2/200      2.33G      1.998      3.013      1.881      1.231          0         94        640: 100% ━━━━━━━━━━━━ 63/63 8.7it/s 7.2s<0.1s
       

In [2]:
# ==============================
# YOLO11s SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLO11s SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11s-seg model...")
model = YOLO("yolo11s-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolo11s_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision (good for RTX 5000 Ada)
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolo11s_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolo11s_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLO11s SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11s-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynam

In [3]:
# ==============================
# YOLO11m SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLO11m SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11m-seg model...")
model = YOLO("yolo11m-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolo11m_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision (good for RTX 5000 Ada)
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolo11m_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolo11m_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLO11m SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11m-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynam

In [4]:
# ==============================
# YOLO11l SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLO11l SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11l-seg model...")
model = YOLO("yolo11l-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolo11l_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision (good for RTX 5000 Ada)
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolo11l_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolo11l_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLO11l SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11l-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynam

In [3]:
# ==============================
# YOLO11x SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLO11x SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11x-seg model...")
model = YOLO("yolo11x-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolo11x_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision (good for RTX 5000 Ada)
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolo11x_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolo11x_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLO11x SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11x-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynam

In [2]:
# ==============================
# YOLO11n SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLO11n SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11/YOLO11sahi"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11n-seg model...")
model = YOLO("yolo11n-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # larger = better for small objects
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolo11n_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,        # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,  # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolo11n_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolo11n_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

print("\n🎯 YOLO11n SAHI-ready model training complete!")

YOLO11n SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11n-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dr

In [1]:
# ==============================
# YOLO11s SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLO11s SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11/YOLO11sahi"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11s-seg model...")
model = YOLO("yolo11s-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # larger = better for small objects
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolo11s_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,        # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,  # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolo11s_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolo11s_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\s✅ Training completed with fallback settings")

print("\n🎯 YOLO11s SAHI-ready model training complete!")

YOLO11s SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11s-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dr

/home/ranjan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Image sizes 960 train, 960 val
Using 2 dataloader workers
Logging results to /home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11/YOLO11sahi/yolo11s_sahi
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/200      8.11G      1.808      2.853      2.105      1.389          0        131        960: 100% ━━━━━━━━━━━━ 63/63 3.4it/s 18.5s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.7it/s 0.9s0.3s
                   all         49        694      0.467      0.576      0.497      0.258      0.443      0.559      0.483      0.224

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      2/200      8.75G      1.545      2.365      1.378      1.208          0        184        960: 100% ━━━━━━━━━━━━ 63/63 4.1it/s 15.4s0.2s
    

In [2]:
# ==============================
# YOLO11m SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLO11m SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11/YOLO11sahi"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11m-seg model...")
model = YOLO("yolo11m-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # larger = better for small objects
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolo11m_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,        # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,  # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolo11m_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolo11m_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\m✅ Training completed with fallback settings")

print("\n🎯 YOLO11m SAHI-ready model training complete!")

YOLO11m SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11m-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.44 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallo

In [3]:
# ==============================
# YOLO11l SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLO11l SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11/YOLO11sahi"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11l-seg model...")
model = YOLO("yolo11l-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # larger = better for small objects
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolo11l_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,        # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,  # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolo11l_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolo11l_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\l✅ Training completed with fallback settings")

print("\n🎯 YOLO11l SAHI-ready model training complete!")

YOLO11l SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11l-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallo

In [6]:
# ==============================
# YOLO11x SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLO11x SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv11/YOLO11sahi"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLO11x-seg model...")
model = YOLO("yolo11x-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # larger = better for small objects
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolo11x_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,        # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,  # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolo11x_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolo11x_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\l✅ Training completed with fallback settings")

print("\n🎯 YOLO11x SAHI-ready model training complete!")

YOLO11x SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLO11x-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallo

In [1]:
# ==============================
# YOLOv8n SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLOv8n SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8n-seg model...")
model = YOLO("yolov8n-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolov8n_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolov8n_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolov8n_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLOv8n SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8n-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation

/home/ranjan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8/yolov8n_seg_training
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/200      2.08G      2.459      3.684      3.091      1.447          0        221        640: 100% ━━━━━━━━━━━━ 63/63 7.3it/s 8.6s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 7.0it/s 0.6s0.2s
                   all         49        694      0.762      0.163      0.249      0.105      0.762      0.163      0.256       0.11

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      2/200      2.22G      1.955      2.983      1.744      1.217          0         94        640: 100% ━━━━━━━━━━━━ 63/63 9.1it/s 6.9s<0.2s
        

In [2]:
# ==============================
# YOLOv8s SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLOv8s SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8s-seg model...")
model = YOLO("yolov8s-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolov8s_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolov8s_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolov8s_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLOv8s SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8s-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation

In [3]:
# ==============================
# YOLOv8m SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLOv8m SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8m-seg model...")
model = YOLO("yolov8m-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolov8m_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolov8m_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolov8m_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLOv8m SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8m-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation

In [4]:
# ==============================
# YOLOv8l SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLOv8l SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8l-seg model...")
model = YOLO("yolov8l-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolov8l_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolov8l_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolov8l_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLOv8l SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8l-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation

In [5]:
# ==============================
# YOLOv8x SEGMENTATION TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("🚀 YOLOv8x SEGMENTATION TRAINING START")
print("=" * 60)

# ------------------------------
# GPU CHECK
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ CUDA GPU not available")

# ------------------------------
# PATHS (EDIT IF NEEDED)
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8x-seg model...")
model = YOLO("yolov8x-seg.pt")
print("✅ Model loaded successfully")

# ------------------------------
# TRAINING CONFIG
# ------------------------------
EPOCHS = 200
IMGSZ = 640
BATCH = 8   # reduce to 4 if you hit OOM
WORKERS = 4

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : GPU (0)")

# ------------------------------
# TRAIN
# ------------------------------
try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,              # GPU
        workers=WORKERS,
        
        project=output_dir,
        name="yolov8x_seg_training",
        exist_ok=True,
        
        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,
        seed=42,
        deterministic=False,
        
        # validation + saving
        val=True,
        save=True,
        save_period=10,
        
        # performance
        amp=True,              # mixed precision
        cache=False,           # set True if dataset fits RAM
        
        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Results saved at: {output_dir}/yolov8x_seg_training")

# ------------------------------
# FALLBACK (OOM / AMP ISSUES)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,          # safer
        device=0,
        workers=0,        # avoids dataloader issues
        
        project=output_dir,
        name="yolov8x_seg_fallback",
        exist_ok=True,
        
        amp=False,        # disable mixed precision
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

🚀 YOLOv8x SEGMENTATION TRAINING START
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8x-seg model...
✅ Model loaded successfully

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 640
   Batch Size  : 8
   Workers     : 4
   Device      : GPU (0)
New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation

In [6]:
# ==============================
# YOLOv8n SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLOv8n SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8n-seg model...")
model = YOLO("yolov8n-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # large resolution helps SAHI slicing
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolov8n_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,         # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,   # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolov8n_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolov8n_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

print("\n🎯 YOLOv8n SAHI-ready model training complete!")

YOLOv8n SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8n-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallo

In [7]:
# ==============================
# YOLOv8s SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLOv8s SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8s-seg model...")
model = YOLO("yolov8s-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # large resolution helps SAHI slicing
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolov8s_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,         # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,   # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolov8s_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolov8s_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

print("\n🎯 YOLOv8s SAHI-ready model training complete!")

YOLOv8s SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8s-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallo

In [8]:
# ==============================
# YOLOv8m SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLOv8m SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8m-seg model...")
model = YOLO("yolov8m-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # large resolution helps SAHI slicing
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolov8m_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,         # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,   # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolov8m_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolov8m_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

print("\n🎯 YOLOv8m SAHI-ready model training complete!")

YOLOv8m SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8m-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallo

In [9]:
# ==============================
# YOLOv8l SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLOv8l SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8l-seg model...")
model = YOLO("yolov8l-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # large resolution helps SAHI slicing
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolov8l_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,         # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,   # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolov8l_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolov8l_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

print("\n🎯 YOLOv8l SAHI-ready model training complete!")

YOLOv8l SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8l-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallo

In [10]:
# ==============================
# YOLOv8x SAHI-READY TRAINING
# ==============================

import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("YOLOv8l SAHI-READY TRAINING (SMALL OBJECT FOCUS)")
print("=" * 60)

# ------------------------------
# GPU SETUP
# ------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = "cpu"
    print("⚠️ No GPU found, using CPU")

# ------------------------------
# PATHS
# ------------------------------
dataset_path = "/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml"
output_dir = "/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}")

print(f"✅ Dataset found: {dataset_path}")

# ------------------------------
# LOAD MODEL
# ------------------------------
print("\n🔄 Loading YOLOv8x-seg model...")
model = YOLO("yolov8x-seg.pt")
print("✅ Model loaded")

# ------------------------------
# TRAINING CONFIG (SAHI-OPTIMIZED)
# ------------------------------
EPOCHS = 200
IMGSZ = 960          # large resolution helps SAHI slicing
BATCH = 8            # reduce if OOM
WORKERS = 2

print("\n📊 Training Configuration:")
print(f"   Epochs      : {EPOCHS}")
print(f"   Image Size  : {IMGSZ}")
print(f"   Batch Size  : {BATCH}")
print(f"   Workers     : {WORKERS}")
print(f"   Device      : {device}")

# ------------------------------
# TRAIN
# ------------------------------
print("\n🚀 Starting training...\n")

try:
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=device,
        workers=WORKERS,

        # output
        project=output_dir,
        name="yolov8x_sahi",
        exist_ok=True,

        # training behavior
        pretrained=True,
        optimizer="auto",
        verbose=True,

        # small-object friendly tweaks
        scale=0.5,         # less aggressive scaling
        mosaic=1.0,
        close_mosaic=10,   # stabilize later epochs

        # validation
        val=True,
        save=True,
        save_period=10,

        # performance
        amp=True if device != "cpu" else False,
        cache=False,

        # logging
        plots=True
    )

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print(f"📁 Saved at: {output_dir}/yolov8x_sahi")

# ------------------------------
# FALLBACK (SAFE MODE)
# ------------------------------
except Exception as e:
    print("\n❌ Training failed:", e)
    print("\n⚠️ Switching to safe fallback config...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=IMGSZ,
        batch=4,
        device=device,
        workers=0,

        project=output_dir,
        name="yolov8x_sahi_fallback",
        exist_ok=True,

        amp=False,
        val=True,
        save=True
    )

    print("\n✅ Training completed with fallback settings")

print("\n🎯 YOLOv8x SAHI-ready model training complete!")

YOLOv8l SAHI-READY TRAINING (SMALL OBJECT FOCUS)
✅ GPU: NVIDIA RTX 5000 Ada Generation
✅ VRAM: 33.80 GB
✅ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml

🔄 Loading YOLOv8x-seg model...
✅ Model loaded

📊 Training Configuration:
   Epochs      : 200
   Image Size  : 960
   Batch Size  : 8
   Workers     : 2
   Device      : 0

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.10.12 torch-2.1.0+cu121 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32237MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallo

In [7]:
import os
import torch
from ultralytics import YOLO
from pathlib import Path
import cv2

# Simple version using Ultralytics plotting
model_path = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/YOLOv8/YOLOv8seg-SAHI/yolov8n_sahi/weights/best.pt'
input_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Images/RGB'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Images/yolov8n/YOLOv8sahi'

os.makedirs(output_dir, exist_ok=True)

# Load model
model = YOLO(model_path)

# Get all images
image_files = []
for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
    image_files.extend(Path(input_dir).glob(f'*{ext}'))
    image_files.extend(Path(input_dir).glob(f'*{ext.upper()}'))

print(f"Found {len(image_files)} images")

# Process each image
for img_path in image_files:
    # Run inference
    results = model(str(img_path), save=False, verbose=False)
    
    # Save visualized result
    for result in results:
        # Save with Ultralytics plotting (includes masks and boxes)
        plotted = result.plot()  # This creates the visualization
        
        # Save to output directory
        output_path = os.path.join(output_dir, f"result_{img_path.name}")
        cv2.imwrite(output_path, plotted)
        
        # Also save the raw masks if needed
        if result.masks is not None:
            # Create mask-only image
            h, w = plotted.shape[:2]
            mask_img = np.zeros((h, w, 3), dtype=np.uint8)
            
            masks = result.masks.data.cpu().numpy() if result.masks.data.is_cuda else result.masks.data.numpy()
            for mask in masks:
                mask_resized = cv2.resize(mask, (w, h))
                mask_binary = (mask_resized > 0.5).astype(np.uint8)
                mask_img[mask_binary == 1] = (0, 255, 0)  # Green masks
            
            mask_path = os.path.join(output_dir, f"mask_{img_path.name}")
            cv2.imwrite(mask_path, mask_img)

print(f"Results saved to: {output_dir}")

Found 55 images
Results saved to: /home/ranjan/Documents/yolo26/earlystageapple/Experiment/Images/yolov8n/YOLOv8sahi
